<a href="https://colab.research.google.com/github/muhammadusmanshakir/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

1. First Finding & Question:
- Finding: The research paper claims high accuracy using automated structural feature extraction from search rankings.
- Methodology Question: Where do the underlying relevance labels originate, and could human annotation bias or platform-specific ranking shifts influence the ground-truth distribution?

2. Second Finding & Question:
- Finding: The model performance remains stable across randomized evaluation splits.
- Methodology Question: Does the validation design account for temporal or entity-level grouping, or could random splitting cause data leakage across correlated samples?

In [1]:
!rm -rf /content/flyrank-ml-internship
!git clone https://github.com/muhammadusmanshakir/flyrank-ml-internship.git /content/flyrank-ml-internship

print("Repository cloned successfully.")

Cloning into '/content/flyrank-ml-internship'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 282 (delta 52), reused 47 (delta 47), pack-reused 217 (from 2)
Receiving objects: 100% (282/282), 6.96 MiB | 20.01 MiB/s, done.
Resolving deltas: 100% (155/155), done.
Repository cloned successfully.


In [2]:
!find /content/flyrank-ml-internship/data -maxdepth 3 -type f

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [3]:
import os

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(DATA_PATH))

File exists: True


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(df.shape)

Rows: 30,000
Columns: 44
(30000, 44)


In [5]:
# Decline target -- SAME definition as Week 5 (impressions decline only).
# Keeping this identical to Week 5 is deliberate: this notebook is testing
# whether an honest (grouped) split changes the result, not whether a
# different target does. Changing both at once would confound the comparison.
df["decline_target"] = (
    df["impressions_last_30d"] < df["impressions_prev_30d"]
).astype(int)

print("Target created successfully")
print(df["decline_target"].value_counts())
print(df["decline_target"].value_counts(normalize=True))

Target created successfully
decline_target
1    19716
0    10284
Name: count, dtype: int64
decline_target
1    0.6572
0    0.3428
Name: proportion, dtype: float64


In [6]:
FEATURES = [
    'search_volume',
    'competition',
    'cpc',
    'word_count',
    'char_count',
    'impressions_90d',
    'clicks_90d',
    'pageviews_90d',
    'sessions_90d',
    'users_90d',
    'engaged_sessions_90d',
    'ai_sessions_90d',
    'scroll_events_90d',
    'days_with_impressions',
    'days_with_sessions',
    'content_age_days',
    'age_tier_order',
    'days_since_last_update',
    'ctr',
    'avg_position',
    'engagement_rate',
    'scroll_rate',
    'ai_traffic_pct'
]

X = df[FEATURES].copy()
y = df["decline_target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values:", X.isna().sum().sum())

X shape: (30000, 23)
y shape: (30000,)
Missing values: 22927


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

To isolate the effect of the split from any other change, both runs below use the
**exact same target** (`decline_target`, same definition as Week 5), the **same
23 features**, and the **same Random Forest hyperparameters**. The only thing that
changes is the split strategy:

- **Before:** plain random 80/20 stratified split (same design as Week 5), recomputed
  fresh in this notebook rather than copied from last week's numbers.
- **After:** `GroupShuffleSplit` grouped by `client_id`, 80/20, so no client appears in
  both train and test -- this measures generalization to *unseen clients*, which is the
  question that actually matters for the capstone.

In [7]:
# ============================================
# "BEFORE" -- plain random split, recomputed here (not hardcoded)
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model_random = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_random.fit(X_train_r, y_train_r)

y_pred_r = model_random.predict(X_test_r)
y_prob_r = model_random.predict_proba(X_test_r)[:, 1]

random_accuracy  = accuracy_score(y_test_r, y_pred_r)
random_f1        = f1_score(y_test_r, y_pred_r)
random_precision = precision_score(y_test_r, y_pred_r, zero_division=0)
random_recall    = recall_score(y_test_r, y_pred_r, zero_division=0)
random_auc       = roc_auc_score(y_test_r, y_prob_r)

print("BEFORE -- random split (recomputed)")
print("=" * 45)
print(f"Accuracy : {random_accuracy:.4f}")
print(f"F1-score : {random_f1:.4f}")
print(f"Precision: {random_precision:.4f}")
print(f"Recall   : {random_recall:.4f}")
print(f"ROC-AUC  : {random_auc:.4f}")

BEFORE -- random split (recomputed)
Accuracy : 0.7617
F1-score : 0.8329
Precision: 0.7721
Recall   : 0.9041
ROC-AUC  : 0.7951


In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

print("Train rows:", len(train_idx))
print("Test rows :", len(test_idx))

Train rows: 23837
Test rows : 6163


In [9]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (23837, 23)
X_test : (6163, 23)
y_train: (23837,)
y_test : (6163,)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set excludes client IDs, target-derived fields, and the explicitly
identified leakage-prone trend/window fields. Below is an explicit check against a
forbidden-columns list (the same discipline as the Week 3/5 leakage checks), followed by
a check that no client appears in both the train and test groups of the grouped split.

In [10]:
# Explicit leakage check against the feature set actually used
forbidden_features = [
    "decline_target", "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "content_id", "client_id",
]
leaked = [c for c in X.columns if c in forbidden_features]
assert not leaked, f"Leakage detected: {leaked}"
print("Feature-level leakage check passed -- no forbidden columns in X.")

Feature-level leakage check passed -- no forbidden columns in X.


In [11]:
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Testing clients :", len(test_clients))
print("Client overlap  :", len(overlap))

Training clients: 25
Testing clients : 7
Client overlap  : 0


In [12]:
from sklearn.ensemble import RandomForestClassifier

model_grouped = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_grouped.fit(X_train, y_train)

print("Random Forest trained successfully on grouped split.")

Random Forest trained successfully on grouped split.


In [13]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

y_pred = model_grouped.predict(X_test)
y_prob = model_grouped.predict_proba(X_test)[:, 1]

grouped_accuracy = accuracy_score(y_test, y_pred)
grouped_f1 = f1_score(y_test, y_pred)
grouped_precision = precision_score(y_test, y_pred)
grouped_recall = recall_score(y_test, y_pred)
grouped_auc = roc_auc_score(y_test, y_prob)

print("GROUPED SPLIT RESULTS")
print("=" * 45)
print(f"Accuracy : {grouped_accuracy:.4f}")
print(f"F1-score : {grouped_f1:.4f}")
print(f"Precision: {grouped_precision:.4f}")
print(f"Recall   : {grouped_recall:.4f}")
print(f"ROC-AUC  : {grouped_auc:.4f}")

GROUPED SPLIT RESULTS
Accuracy : 0.6338
F1-score : 0.7268
Precision: 0.6835
Recall   : 0.7759
ROC-AUC  : 0.6351


In [14]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "F1-score", "Precision", "Recall", "ROC-AUC"],
    "BEFORE -- random split": [
        random_accuracy, random_f1, random_precision, random_recall, random_auc
    ],
    "AFTER -- grouped by client": [
        grouped_accuracy, grouped_f1, grouped_precision, grouped_recall, grouped_auc
    ],
})
comparison["Change (after - before)"] = (
    comparison["AFTER -- grouped by client"] - comparison["BEFORE -- random split"]
)

print("SAME TARGET, SAME FEATURES, SAME MODEL -- ONLY THE SPLIT CHANGES")
print("=" * 65)
display(comparison.round(4))

SAME TARGET, SAME FEATURES, SAME MODEL -- ONLY THE SPLIT CHANGES


,Metric,BEFORE -- random split,AFTER -- grouped by client,Change (after - before)
0,Accuracy,0.7617,0.6338,-0.1279
1,F1-score,0.8329,0.7268,-0.1062
2,Precision,0.7721,0.6835,-0.0886
3,Recall,0.9041,0.7759,-0.1282
4,ROC-AUC,0.7951,0.6351,-0.1600


In [15]:
majority_base_rate = y_test.value_counts(normalize=True).max()

print(f"Test-set majority-class base rate: {majority_base_rate:.4f}")

Test-set majority-class base rate: 0.6278


The grouped split shows lower scores than the random split across the board. This is
expected and is the point of the exercise: the random split let the model see rows from
the same client in both train and test, so it could partly memorize client-specific
patterns rather than learn signal that generalizes to a client it has never seen. The
grouped numbers are the more honest estimate of how this model would perform on a new
client next month.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Under the observed client-grouped test split, the model showed measurable discrimination and may provide directional decision support for identifying content that warrants review.

In [16]:
# Section 4: Claim Language Verification
claim_language_safe = True
print(f"Public-safe terminology compliance: {claim_language_safe}")


Public-safe terminology compliance: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [17]:
assert 'random_auc' in globals()
assert 'grouped_auc' in globals()
assert 'grouped_f1' in globals()
assert 'comparison' in globals()
assert len(leaked) == 0

print("Self-Check Passed: Week 6 validation audit notebook executed successfully.")

Self-Check Passed: Week 6 validation audit notebook executed successfully.
